# 4.1. Metrics: STA (Style Transfer Accuracy)

Acknowledgement:

- Adopted from https://www.kaggle.com/code/rhodiumbeng/classifying-multi-label-comments-0-9741-lb/notebook


In [29]:
EXTERNAL_PATH = "../data/external/"
JIGSAW_DATASET = "jigsaw-toxic-comment-classification-challenge"

In [30]:
# Downloading the data from kaggle
import kaggle

kaggle.api.competition_download_files(JIGSAW_DATASET, path=EXTERNAL_PATH, quiet=False)

jigsaw-toxic-comment-classification-challenge.zip: Skipping, found more recently modified local copy (use --force to force download)


In [31]:
# Unzipping the data
import os
import zipfile

DATAPATH = EXTERNAL_PATH + JIGSAW_DATASET

with zipfile.ZipFile(DATAPATH + ".zip", "r") as zip_ref:
    zip_ref.extractall(DATAPATH)

    # Unzip extracted files
    for file in os.listdir(DATAPATH):
        if not file.endswith(".zip"):
            continue

        with zipfile.ZipFile(DATAPATH + "/" + file, "r") as zip_ref:
            zip_ref.extractall(DATAPATH)

        os.remove(DATAPATH + "/" + file)

In [32]:
import numpy as np
import pandas as pd
import re

from matplotlib import pyplot as plt

In [33]:
train_df = pd.read_csv(DATAPATH + "/train.csv")
test_df = pd.read_csv(DATAPATH + "/test.csv")

In [34]:
train_df.describe()

,toxic,severe_toxic,obscene,threat,insult,identity_hate
count,159571.000000,159571.000000,159571.000000,159571.000000,159571.000000,159571.000000
mean,0.095844,0.009996,0.052948,0.002996,0.049364,0.008805
std,0.294379,0.099477,0.223931,0.054650,0.216627,0.093420
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [35]:
cols_target = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

In [36]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"what's", "what is ", text)
    text = re.sub(r"\'s", " ", text)
    text = re.sub(r"\'ve", " have ", text)
    text = re.sub(r"can't", "cannot ", text)
    text = re.sub(r"n't", " not ", text)
    text = re.sub(r"i'm", "i am ", text)
    text = re.sub(r"\'re", " are ", text)
    text = re.sub(r"\'d", " would ", text)
    text = re.sub(r"\'ll", " will ", text)
    text = re.sub(r"\'scuse", " excuse ", text)
    text = re.sub("\W", " ", text)
    text = re.sub("\s+", " ", text)
    text = text.strip(" ")
    return text

In [37]:
# Cleaning the comments in the train dataset
train_df["comment_text"] = train_df["comment_text"].map(lambda x: clean_text(x))

In [38]:
# Cleaning the comments in the test dataset
test_df["comment_text"] = test_df["comment_text"].map(lambda x: clean_text(x))

In [39]:
X = train_df.comment_text
X_test = test_df.comment_text

In [40]:
# Import and instantiate TfidfVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

vec = TfidfVectorizer(max_features=5000, stop_words="english")
vec

TfidfVectorizer(max_features=5000, stop_words='english')

In [41]:
# Learn the vocabulary in the training data. Use this vocabulary to create a document-term matrix.
X_dtm = vec.fit_transform(X)

# Examine the created document-term matrix from X_train.
X_dtm

<159571x5000 sparse matrix of type '<class 'numpy.float64'>'
	with 3178792 stored elements in Compressed Sparse Row format>

In [42]:
# Save the vectorizer
import pickle

with open("../models/vectorizer.pkl", "wb") as f:
    pickle.dump(vec, f)

In [43]:
# Transform the test data, using the pre-fitted vocabulary, into a document-term matrix
X_test_dtm = vec.transform(X_test)

# Examine the resulting document-term matrix of X_test
X_test_dtm

<153164x5000 sparse matrix of type '<class 'numpy.float64'>'
	with 2618972 stored elements in Compressed Sparse Row format>

## Binary Relevance


In [44]:
import pickle

from copy import deepcopy

In [45]:
submission_binary = pd.read_csv(DATAPATH + "/sample_submission.csv")

In [46]:
# Import and set up the Logistic Regression model
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

logreg = LogisticRegression(C=12.0, max_iter=1000)
binary_logreg = {}

for label in cols_target:
    print(f'Processing "{label}" column...')
    y = train_df[label]

    # Train the model using the designated training data, X_dtm and y
    logreg.fit(X_dtm, y)

    # Put the trained model into the dictionary
    binary_logreg[label] = deepcopy(logreg)

    # Calculate and print the training accuracy by predicting outcomes for training data, X_dtm
    y_pred_X = logreg.predict(X_dtm)
    print(f" - Training accuracy is {accuracy_score(y, y_pred_X)}")

    # Compute the probability predictions for the testing data, X_test_dtm
    test_y_prob = logreg.predict_proba(X_test_dtm)[:, 1]
    submission_binary[label] = test_y_prob

# Save trained models
with open("../models/binary_logreg.pkl", "wb") as f:
    pickle.dump(binary_logreg, f)

Processing "toxic" column...
 - Training accuracy is 0.9639909507366626
Processing "severe_toxic" column...
 - Training accuracy is 0.9920850279812748
Processing "obscene" column...
 - Training accuracy is 0.9832425691385026
Processing "threat" column...
 - Training accuracy is 0.9981199591404453
Processing "insult" column...
 - Training accuracy is 0.9755469352200588
Processing "identity_hate" column...
 - Training accuracy is 0.9939650688408295


## Classifier Chains


In [47]:
submission_chains = pd.read_csv(DATAPATH + "/sample_submission.csv")

In [48]:
from scipy.sparse import csr_matrix, hstack


def add_feature(X, feature_to_add):
    """
    Returns sparse feature matrix with added feature.
    feature_to_add can also be a list of features.
    """
    return hstack([X, csr_matrix(feature_to_add).T], "csr")

In [49]:
chains_logreg = {}

for label in cols_target:
    print(f"Processing {label} column...")
    y = train_df[label]

    # Train the model using X_dtm & y
    logreg.fit(X_dtm, y)

    # Put the trained model into the dictionary
    chains_logreg[label] = deepcopy(logreg)

    # Compute the training accuracy
    y_pred_X = logreg.predict(X_dtm)
    print(f" - Training Accuracy is {accuracy_score(y, y_pred_X)}")

    # Make predictions from test_X
    test_y = logreg.predict(X_test_dtm)
    test_y_prob = logreg.predict_proba(X_test_dtm)[:, 1]
    submission_chains[label] = test_y_prob

    # Chain current label to X_dtm
    X_dtm = add_feature(X_dtm, y)
    print(f"Shape of X_dtm is now {X_dtm.shape}")

    # Chain current label predictions to test_X_dtm
    X_test_dtm = add_feature(X_test_dtm, test_y)
    print(f"Shape of test_X_dtm is now {X_test_dtm.shape}")

# Save trained models
with open("../models/chains_logreg.pkl", "wb") as f:
    pickle.dump(chains_logreg, f)

Processing toxic column...
 - Training Accuracy is 0.9639909507366626
Shape of X_dtm is now (159571, 5001)
Shape of test_X_dtm is now (153164, 5001)
Processing severe_toxic column...
 - Training Accuracy is 0.9926239730276805
Shape of X_dtm is now (159571, 5002)
Shape of test_X_dtm is now (153164, 5002)
Processing obscene column...
 - Training Accuracy is 0.9853043472811476
Shape of X_dtm is now (159571, 5003)
Shape of test_X_dtm is now (153164, 5003)
Processing threat column...
 - Training Accuracy is 0.9984395660865696
Shape of X_dtm is now (159571, 5004)
Shape of test_X_dtm is now (153164, 5004)
Processing insult column...
 - Training Accuracy is 0.9826973572892318
Shape of X_dtm is now (159571, 5005)
Shape of test_X_dtm is now (153164, 5005)
Processing identity_hate column...
 - Training Accuracy is 0.9956132379943724
Shape of X_dtm is now (159571, 5006)
Shape of test_X_dtm is now (153164, 5006)


## Combined score


In [50]:
submission_combined = pd.read_csv(DATAPATH + "/sample_submission.csv")

In [51]:
# Using average of probabilities from binary and chain classifiers
submission_combined[cols_target] = (
    submission_binary[cols_target] + submission_chains[cols_target]
) / 2
submission_combined.head()

,id,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,00001cee341fdb12,0.999956,0.450019,0.999828,0.066352,0.914937,0.493081
1,0000247867823ef7,0.002396,0.000216,0.000314,0.000202,0.003252,0.000259
2,00013b17ad220c46,0.011099,0.000059,0.002078,0.000024,0.004897,0.000902
3,00017563c3f7919a,0.001326,0.001081,0.001035,0.000090,0.000646,0.000020
4,00017695ad8997eb,0.019311,0.000416,0.000786,0.000382,0.002061,0.000332


In [52]:
submission_combined.to_csv("../data/temporary/submission_combined.csv", index=False)

This model scored 0.96087 on the public leaderboard and 0.96318 on the private leaderboard.


## Inference test


In [62]:
# Load vectorizer and models
with open("../models/vectorizer.pkl", "rb") as f:
    vec = pickle.load(f)

with open("../models/binary_logreg.pkl", "rb") as f:
    binary_logreg = pickle.load(f)

with open("../models/chains_logreg.pkl", "rb") as f:
    chains_logreg = pickle.load(f)

In [65]:
# Inference test
prompt_toxic = "I hate you stupid fucking face!"
prompt_not_toxic = "Australia is beatiful!"

input_data = [prompt_toxic, prompt_not_toxic]

In [66]:
def toxicity_report(input_data):
    """
    Returns a dataframe with toxicity probabilities for each label.
    """
    input_data = [clean_text(x) for x in input_data]
    input_data = vec.transform(input_data)

    # Create empty dataframe
    df = pd.DataFrame(columns=cols_target)

    # Predict using binary classifier
    for label in cols_target:
        prob = binary_logreg[label].predict_proba(input_data)[:, 1]
        df[label] = prob

    # Chain predictions
    for label in cols_target:
        y = chains_logreg[label].predict(input_data)
        prob = chains_logreg[label].predict_proba(input_data)[:, 1]

        df[label] = (df[label] + prob) / 2

        input_data = add_feature(input_data, y)

    return df


toxicity_report(input_data)

,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,1.000000,0.417646,0.999406,0.304330,0.992161,0.383265
1,0.014986,0.000274,0.005063,0.000237,0.001892,0.000181
